# EMBEL HLWM 8B — resumable dual-T4 training

Use **Save Version → Save & Run All** with **GPU T4 x2** and Internet enabled.
Attach the `hlwm8b-kaggle-code` and `hlwm-beast-teacher-snapshot` datasets. The
run stops early enough to checkpoint and finish cleanly. On the next version it
resumes from a checkpoint attached as a dataset, or automatically from a private
Hugging Face repo when Kaggle secrets `HF_TOKEN` and `HF_REPO` are configured.

Main training stays locked until the snapshot contains at least 25,000 accepted
multi-teacher records. A small snapshot can still run the real-Qwen preflight.


In [ ]:
import os, platform, subprocess, sys, torch
print(platform.platform())
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.device_count() == 2, f'Choose GPU T4 x2; found {torch.cuda.device_count()} GPU(s)'
print('python', sys.version)


In [ ]:
!python -m pip install -q 'transformers==4.56.2' 'accelerate==1.10.1' 'peft==0.17.1' 'bitsandbytes>=0.46,<0.49' 'safetensors==0.6.2' 'sentencepiece==0.2.1' 'pytest==8.4.1' 'huggingface_hub>=0.34,<1'


In [ ]:
from pathlib import Path
import json, shutil, zipfile

WORK = Path('/kaggle/working/hlwm8b-beast')
WORK.mkdir(parents=True, exist_ok=True)
code_archives = list(Path('/kaggle/input').rglob('hlwm8b-kaggle-code.zip'))
if len(code_archives) != 1:
    raise FileNotFoundError('Attach exactly one hlwm8b-kaggle-code dataset.')
CODE_ROOT = WORK / 'code'
if CODE_ROOT.exists(): shutil.rmtree(CODE_ROOT)
with zipfile.ZipFile(code_archives[0]) as archive: archive.extractall(CODE_ROOT)
PROJECT = CODE_ROOT / 'hlwm8b_kaggle'

data_archives = list(Path('/kaggle/input').rglob('hlwm-beast-teacher-snapshot.zip'))
if len(data_archives) != 1:
    raise FileNotFoundError('Attach exactly one hlwm-beast-teacher-snapshot dataset.')
DATA_ROOT = WORK / 'data-snapshot'
if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
with zipfile.ZipFile(data_archives[0]) as archive: archive.extractall(DATA_ROOT)
manifest_paths = list(DATA_ROOT.rglob('manifest.json'))
if len(manifest_paths) != 1: raise RuntimeError('Teacher snapshot manifest is missing or ambiguous.')
DATA = manifest_paths[0].parent
MANIFEST = json.loads(manifest_paths[0].read_text())
assert MANIFEST['name'] == 'hlwm-beast-teacher-snapshot'
print(json.dumps({'project': str(PROJECT), 'data': str(DATA), 'manifest': MANIFEST}, indent=2))


In [ ]:
# Optional zero-touch persistence through a private Hugging Face model repository.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for name in ('HF_TOKEN', 'HF_REPO'):
        try:
            value = secrets.get_secret(name)
            if value: os.environ[name] = value
        except Exception:
            pass
except Exception:
    pass
print('automatic remote resume:', bool(os.getenv('HF_TOKEN') and os.getenv('HF_REPO')))


In [ ]:
# Reconstruct a numbered resume archive when it was uploaded as Kaggle data.
OUTPUT = Path('/kaggle/working/hlwm8b-output')
OUTPUT.mkdir(parents=True, exist_ok=True)
parts_manifests = list(Path('/kaggle/input').rglob('hlwm8b-checkpoint-*.zip.parts.json'))
for parts_manifest in parts_manifests:
    spec = json.loads(parts_manifest.read_text())
    archive_path = OUTPUT / spec['archive']
    with archive_path.open('wb') as destination:
        for part in spec['parts']:
            matches = list(parts_manifest.parent.rglob(part['name']))
            if len(matches) != 1: raise FileNotFoundError(part['name'])
            with matches[0].open('rb') as source: shutil.copyfileobj(source, destination)
    import hashlib
    digest_object = hashlib.sha256()
    with archive_path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''): digest_object.update(chunk)
    digest = digest_object.hexdigest()
    if digest != spec['archive_sha256']: raise ValueError('Reconstructed resume archive checksum mismatch')
    checkpoint_dir = OUTPUT / Path(spec['archive']).stem
    if checkpoint_dir.exists(): shutil.rmtree(checkpoint_dir)
    checkpoint_dir.mkdir()
    with zipfile.ZipFile(archive_path) as archive: archive.extractall(checkpoint_dir)
    archive_path.unlink()
    print('reconstructed', checkpoint_dir)
for archive_path in Path('/kaggle/input').rglob('hlwm8b-checkpoint-*.zip'):
    checkpoint_dir = OUTPUT / archive_path.stem
    if checkpoint_dir.exists(): continue
    checkpoint_dir.mkdir()
    with zipfile.ZipFile(archive_path) as archive: archive.extractall(checkpoint_dir)
    print('imported full checkpoint archive', checkpoint_dir)


In [ ]:
import py_compile, subprocess
for path in PROJECT.glob('*.py'):
    py_compile.compile(str(path), doraise=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(PROJECT/'test_hlwm8b.py')], cwd=PROJECT, check=True)
print('Static checks and HLWM unit tests passed')


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id='Qwen/Qwen3-8B', revision='b968826d9c46dd6066d109eabc6255188de91218')
print('Pinned Qwen3-8B snapshot cached')


## Exact real-Qwen preflight

This performs a real 4-bit Qwen3-8B forward/backward pass before the two-GPU run.
It is allowed on a small snapshot, but it never bypasses the main data-readiness gate.


In [ ]:
PREFLIGHT = Path('/kaggle/working/hlwm8b-preflight')
if PREFLIGHT.exists(): shutil.rmtree(PREFLIGHT)
preflight = [sys.executable, str(PROJECT/'train_hlwm8b.py'),
    '--data-dir', str(DATA), '--output-dir', str(PREFLIGHT), '--preflight-only', '--allow-small-data',
    '--resume', 'off', '--batch-size', '1', '--gradient-accumulation', '1', '--num-workers', '0',
    '--joint-updates', '1', '--on-policy-updates', '0', '--max-prompt-tokens', '384', '--max-answer-tokens', '192']
environment = os.environ.copy(); environment['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.run(preflight, cwd=PROJECT, env=environment, check=True)
print(json.dumps(json.loads((PREFLIGHT/'preflight.json').read_text()), indent=2))


## Pausable two-T4 conversion

Both T4s train one distributed 8B model. The run saves every 50 updates, reacts to
`/kaggle/working/HLWM_STOP`, and stops 20 minutes before its 8.5-hour budget. Re-run
this notebook with the checkpoint output attached, or configure HF secrets for
automatic remote resume.


In [ ]:
if not MANIFEST.get('ready_for_main_training'):
    raise RuntimeError(f"Data snapshot has {MANIFEST.get('accepted_teacher_records', 0)} accepted teacher records; main training requires {MANIFEST.get('minimum_teacher_records', 25000)}. Keep the local teacher factory running, export again, and replace the Kaggle data snapshot.")

command = ['accelerate', 'launch', '--multi_gpu', '--num_processes', '2', '--mixed_precision', 'fp16',
    str(PROJECT/'train_hlwm8b.py'), '--data-dir', str(DATA), '--output-dir', str(OUTPUT),
    '--input-root', '/kaggle/input', '--resume', 'auto', '--joint-updates', '1600',
    '--on-policy-updates', '200', '--batch-size', '1', '--gradient-accumulation', '8',
    '--learning-rate', '0.00008', '--sidecar-learning-rate', '0.00016',
    '--save-every-updates', '50', '--eval-every-updates', '100', '--keep-checkpoints', '2',
    '--max-runtime-hours', '8.5', '--runtime-save-buffer-minutes', '20',
    '--max-prompt-tokens', '384', '--max-answer-tokens', '192', '--num-workers', '2']
if os.getenv('HF_REPO'): command.extend(['--hf-repo', os.environ['HF_REPO']])
subprocess.run(command, cwd=PROJECT, check=True)
RUN_STATUS = json.loads((OUTPUT/'run-status.json').read_text())
TRAINING_COMPLETE = RUN_STATUS['status'] == 'training_complete'
print(json.dumps({'status': RUN_STATUS['status'], 'global_update': RUN_STATUS['global_update'], 'pause_reason': RUN_STATUS.get('pause_reason')}, indent=2))


In [ ]:
sys.path.insert(0, str(PROJECT)) if str(PROJECT) not in sys.path else None
from checkpointing_hlwm8b import find_resume_checkpoint
CHECKPOINT = find_resume_checkpoint(OUTPUT)
if CHECKPOINT is None: raise RuntimeError('No verified checkpoint was produced')
EVALUATION = OUTPUT / 'evaluation'
if TRAINING_COMPLETE:
    if EVALUATION.exists(): shutil.rmtree(EVALUATION)
    evaluation = [sys.executable, str(PROJECT/'evaluate_hlwm8b.py'), '--checkpoint', str(CHECKPOINT),
        '--data-dir', str(DATA), '--output-dir', str(EVALUATION), '--validation-samples', '64',
        '--test-samples', '64', '--max-new-tokens', '96']
    environment = os.environ.copy(); environment['CUDA_VISIBLE_DEVICES'] = '0'
    subprocess.run(evaluation, cwd=PROJECT, env=environment, check=True)
else:
    print('Training paused safely. Evaluation will run automatically after a later resumed session completes.')


In [ ]:
DOWNLOADS = Path('/kaggle/working/hlwm8b-downloads')
if DOWNLOADS.exists(): shutil.rmtree(DOWNLOADS)
package = [sys.executable, str(PROJECT/'package_hlwm8b.py'), '--checkpoint', str(CHECKPOINT),
    '--output-dir', str(DOWNLOADS), '--code-dir', str(PROJECT), '--data-manifest', str(DATA/'manifest.json')]
if TRAINING_COMPLETE: package.extend(['--evaluation-dir', str(EVALUATION)])
subprocess.run(package, cwd=PROJECT, check=True)
download_spec = json.loads((DOWNLOADS/'downloads.json').read_text())
print(json.dumps(download_spec, indent=2))


In [ ]:
from IPython.display import FileLink, display
files = [DOWNLOADS/'downloads.json']
for manifest in DOWNLOADS.glob('*.parts.json'):
    files.append(manifest)
    spec = json.loads(manifest.read_text())
    files.extend(DOWNLOADS/part['name'] for part in spec['parts'])
for path in files:
    if path.exists(): display(FileLink(str(path)))
print('If status was paused, attach every checkpoint part plus its .parts.json to the next Kaggle run unless HF resume is configured.')
